# Hair 6클래스 경량 모델: B0·256·증강

이미 완료된 네 실험 중 두 B0·256 모델의 **Validation Macro F1**을 비교해 증강 모델을 경량 후보로 고정했습니다. 이 노트북은 새 학습 없이 고정 Test에서 한 번 평가하고 별도 모델 ZIP을 만듭니다. B1·384 성능 우선 ZIP은 유지합니다.

이 노트북은 경량 후보의 Test 결과를 본 뒤 모델 조건을 다시 고르는 용도가 아닙니다. 실제 장비 속도와 메모리 측정은 별도 단계입니다.


<!-- mediflow-notebook-guide-v1 -->
### 실행 안내

- **이 노트북의 목적:** 이미 학습된 Hair 6클래스 경량 후보를 검증·패키징합니다.
- **실행 순서:** Colab에서 첫 셀부터 아래로 차례대로 실행합니다. 중간 셀만 단독 실행하면 앞에서 만든 설정과 코드가 없어 오류가 날 수 있습니다.
- **수정할 곳:** 앞부분의 경로·실험 설정 셀을 먼저 확인합니다. 내장 실행 코드 셀은 펼쳐서 읽을 수 있지만 보통 수정하지 않습니다.
- **결과 확인:** 끝부분의 저장 경로와 보고서 출력에서 결과를 확인합니다. ZIP과 학습 결과를 혼동하지 않도록 각 단계의 설명을 읽어 주세요.


## 1. Drive 경로

기존 실험 결과와 새 6클래스 데이터 ZIP을 그대로 읽습니다.


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/mediflow_Project'
PARENT_RUN_DIR = (
    PROJECT_ROOT + '/2_results/hair/'
    'hair_six_class_four_20260928_011824_ad21e8ae'
)
BATCH_SIZE = 32


## 2. 실행 환경

GPU를 사용할 수 있으면 평가가 더 빠릅니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2
import tensorflow as tf
import keras
if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('런타임을 다시 시작하고 처음부터 실행하세요.')


## 3. 고정 코드

원본 실행 코드를 노트북에 포함했습니다. 수정하지 마세요.


In [ ]:
from pathlib import Path
_EMBEDDED_SOURCE_DIR = Path('/content/mediflow_notebook_sources/19_hair_six_class_light_package_colab')
_EMBEDDED_SOURCE_DIR.mkdir(parents=True, exist_ok=True)


<!-- mediflow-embedded-module-v1 -->
### 내장 실행 코드: `common_engine.py`

이 셀의 코드는 줄 단위로 읽을 수 있도록 펼쳐 놓았습니다. Colab 임시 공간에 기록한 뒤 아래 실행 셀에서 원본과 동일한 SHA-256을 확인합니다. 설정 변경은 앞쪽 설정 셀에서 하세요.


In [ ]:
%%writefile /content/mediflow_notebook_sources/19_hair_six_class_light_package_colab/common_engine.py
"""Sequential domain-configured experiments; validation selection precedes any test inference.

The Colab notebook embeds an exact copy of this module so no repository checkout
is needed in Colab. Completed trials are reused only after artifact verification.
Interrupted attempts are preserved and restarted, not resumed mid-epoch.
"""

from __future__ import annotations

import csv
import hashlib
import json
import time
import uuid
from pathlib import Path

import keras
import numpy as np

PROTOCOL = "mediflow_common_v1"
TRIALS = [
    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},
    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},
    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},
    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},
    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},
]


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def classification_metrics(truth, probabilities, count=5):
    truth = np.asarray(truth, dtype=np.int64)
    probabilities = np.asarray(probabilities)
    if (
        truth.ndim != 1
        or not len(truth)
        or probabilities.shape != (len(truth), count)
        or not np.isfinite(probabilities).all()
        or np.any(truth < 0)
        or np.any(truth >= count)
    ):
        raise ValueError("Invalid evaluation arrays")
    predictions = probabilities.argmax(axis=1)
    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)
    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(count),
        where=precision + recall != 0,
    )
    return {
        "accuracy": float(np.mean(truth == predictions)),
        "macro_f1": float(f1.mean()),
        "class_f1": f1.tolist(),
        "precision": precision.tolist(),
        "recall": recall.tolist(),
        "support": cm.sum(1).tolist(),
        "confusion_matrix": cm.tolist(),
        "count": len(truth),
    }


def predict_dataset(model, dataset):
    truth, probabilities = [], []
    for images, labels in dataset:
        probabilities.extend(model(images, training=False).numpy())
        truth.extend(np.argmax(labels.numpy(), axis=1))
    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)


def save_predictions(path, paths, truth, probabilities):
    if len(paths) != len(truth):
        raise ValueError("File order and prediction count differ")
    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.writer(stream)
        writer.writerow(
            [
                "path",
                "true_index",
                "pred_index",
                *[f"prob_C{i}" for i in range(probabilities.shape[1])],
            ]
        )
        for name, target, probs in zip(paths, truth, probabilities, strict=True):
            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])


def loss_function(name):
    if name == "ce":
        return keras.losses.CategoricalCrossentropy()
    if name == "ls005":
        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
    if name == "focal15":
        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)
    raise ValueError(name)


def build_model(spec):
    builder = {
        "B0": keras.applications.EfficientNetB0,
        "B1": keras.applications.EfficientNetB1,
        "V2S": keras.applications.EfficientNetV2S,
    }
    size = spec["size"]
    backbone = builder[spec["backbone"]](
        include_top=False, weights="imagenet", input_shape=(size, size, 3)
    )
    backbone.trainable = False
    inputs = keras.Input((size, size, 3))
    features = backbone(inputs, training=False)
    features = keras.layers.GlobalAveragePooling2D()(features)
    features = keras.layers.Dropout(0.3)(features)
    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=loss_function(spec["loss"]),
        metrics=["accuracy"],
    )
    return model


def configure_partial(model, loss_name):
    backbones = [
        layer
        for layer in model.layers
        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()
    ]
    if len(backbones) != 1:
        raise ValueError("Expected one EfficientNet backbone")
    backbone = backbones[0]
    backbone.trainable = True
    for index, layer in enumerate(backbone.layers):
        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(
            layer, keras.layers.BatchNormalization
        )
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]
    )
    return [layer.name for layer in backbone.layers if layer.trainable]


class HistoryBackup(keras.callbacks.Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.values = {}

    def on_epoch_end(self, epoch, logs=None):
        for key, value in (logs or {}).items():
            self.values.setdefault(key, []).append(float(value))
        write_json(self.path, self.values)


def fit_stage(model, train, val, directory, name, epochs):
    best = directory / (name + "_best.keras")
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            str(best), monitor="val_accuracy", mode="max", save_best_only=True
        ),
        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),
        HistoryBackup(directory / (name + "_history.json")),
        keras.callbacks.TerminateOnNaN(),
    ]
    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)
    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}
    if len(values.get("val_accuracy", [])) != epochs or not all(
        np.isfinite(seq).all() for seq in values.values()
    ):
        raise RuntimeError("Incomplete or non-finite training; attempt retained")
    model.save(directory / (name + "_last.keras"))
    return values, best


def checkpoint_choice(baseline_score, new_score):
    """Keep the earlier/simpler checkpoint on ties."""
    return new_score > baseline_score


def cached_record(root, trial_id, signature):
    trial_dir = Path(root) / trial_id
    marker = trial_dir / "completed.json"
    if not marker.exists():
        return None
    record = read_json(marker)
    if record["signature"] != signature:
        raise ValueError("Resume settings differ; use a new suite directory")
    for relative, digest in record["artifact_hashes"].items():
        target = (trial_dir / relative).resolve()
        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:
            raise ValueError("Completed artifact changed or corrupted: " + relative)
    return record


def evaluate_to_files(model, dataset, paths, directory, prefix):
    truth, probabilities = predict_dataset(model, dataset)
    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])
    write_json(directory / (prefix + "_metrics.json"), metrics)
    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)
    return metrics


def run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):
    cached = cached_record(root, spec["id"], signature)
    if cached:
        return cached
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])
    directory.mkdir(parents=True, exist_ok=False)
    write_json(directory / "spec.json", spec)
    train, _ = dataset_factory("train", spec["size"], True)
    val, paths = dataset_factory("val", spec["size"], False)
    started = time.monotonic()
    model = build_model(spec)
    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)
    del model
    keras.backend.clear_session()
    model = keras.models.load_model(best1, compile=False)
    trainable = []
    h2 = {key: [] for key in h1}
    best2 = best1
    if epochs2:
        trainable = configure_partial(model, spec["loss"])
        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)
    del model
    selected_stage = (
        "stage2"
        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))
        else "stage1"
    )
    selected = best2 if selected_stage == "stage2" else best1
    model = keras.models.load_model(selected, compile=False)
    metrics = evaluate_to_files(model, val, paths, directory, "validation")
    record = {
        "id": spec["id"],
        "spec": spec,
        "signature": signature,
        "attempt": directory.name,
        "selected_model": selected.name,
        "selected_stage": selected_stage,
        "validation": metrics,
        "stage1_best_val": max(h1["val_accuracy"]),
        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,
        "training_seconds": time.monotonic() - started,
        "parameters": model.count_params(),
        "model_bytes": selected.stat().st_size,
        "trainable_backbone_layers": trainable,
        "history": {key: h1[key] + h2[key] for key in h1},
        "stage_boundary": len(h1["accuracy"]),
    }
    finish_record(directory, record)
    return record


def finish_record(directory, record):
    write_json(directory / "record.json", record)
    record["artifact_hashes"] = {
        str(path.relative_to(directory.parent)): file_hash(path)
        for path in directory.iterdir()
        if path.is_file()
    }
    write_json(directory.parent / "completed.json", record)


def selected_model_path(root, record):
    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]


def extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):
    trial_id = "b1_256_ls005_extend5"
    cached = cached_record(root, trial_id, signature)
    if cached:
        return cached
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])
    directory.mkdir(parents=True, exist_ok=False)
    parent_dir = Path(root) / parent["id"] / parent["attempt"]
    # Continue from epoch 10's LAST checkpoint, including optimizer state.
    source = parent_dir / "stage2_last.keras"
    model = keras.models.load_model(source)
    if model.optimizer is None:
        raise ValueError("Extension requires saved optimizer")
    train, _ = dataset_factory("train", 256, True)
    val, paths = dataset_factory("val", 256, False)
    started = time.monotonic()
    history, best = fit_stage(model, train, val, directory, "extension", epochs)
    del model
    parent_best = selected_model_path(root, parent)
    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])
    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))
    selected = directory / "selected.keras"
    import shutil

    shutil.copyfile(best if keep_extension else parent_best, selected)
    model = keras.models.load_model(selected, compile=False)
    metrics = evaluate_to_files(model, val, paths, directory, "validation")
    record = {
        "id": trial_id,
        "spec": {**parent["spec"], "id": trial_id},
        "signature": signature,
        "attempt": directory.name,
        "selected_model": selected.name,
        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],
        "validation": metrics,
        "parameters": model.count_params(),
        "model_bytes": selected.stat().st_size,
        "training_seconds": time.monotonic() - started,
        "history": {key: parent["history"][key] + history[key] for key in history},
        "stage_boundary": parent["stage_boundary"],
        "extension_boundary": len(parent["history"]["accuracy"]),
        "parent_last_sha256": file_hash(source),
        "optimizer_restored": True,
        "extension_best_val": max(history["val_accuracy"]),
        "stage1_best_val": parent["stage1_best_val"],
        "stage2_best_val": parent["stage2_best_val"],
    }
    finish_record(directory, record)
    return record


def select_winner(records):
    # Stable order preserves earlier experiments on exact ties.
    return max(records, key=lambda record: record["validation"]["accuracy"])


<!-- mediflow-embedded-module-v1 -->
### 내장 실행 코드: `hair_six_class_light.py`

이 셀의 코드는 줄 단위로 읽을 수 있도록 펼쳐 놓았습니다. Colab 임시 공간에 기록한 뒤 아래 실행 셀에서 원본과 동일한 SHA-256을 확인합니다. 설정 변경은 앞쪽 설정 셀에서 하세요.


In [ ]:
%%writefile /content/mediflow_notebook_sources/19_hair_six_class_light_package_colab/hair_six_class_light.py
"""Evaluate and package the fixed six-class Hair B0 lightweight alternate."""

from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

RUN_NAME = "hair_six_class_four_20260928_011824_ad21e8ae"
DATA_NAME = "hair_clean_v2_6class_d6ac77b7d507.zip"
DATA_SHA256 = "e2a6b4347d94bd7050d17bd3aca404161693083b12afb51d7e20aa9b284f0f93"
TRIAL = "b0_256_augmented"
MODEL_SHA256 = "38e91fee1c514b75165d07f5dfbb0c52fdb7ac92313b40d47abb50c71367d432"
HEAVY_SHA256 = "5ee54f51e65d8656fc81e88de96ad82a6d119991aa74684f921e4cd7e8b7efbb"
CLASSES = ["모낭사이홍반", "미세각질", "비듬", "탈모", "피지과다", "양호"]
OUTPUT_NAME = "public_candidate_6class_light_b0_256_augmented_v1"


def _read(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def _sha(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _check(ok: bool, message: str) -> None:
    if not ok:
        raise ValueError(message)


def verify_candidate(parent: Path) -> dict:
    _check(parent.is_dir() and parent.name == RUN_NAME, "6클래스 실험 폴더가 다릅니다.")
    names = _read(parent / "class_names.json")
    config = _read(parent / "run_config.json")
    contract = _read(parent / "dataset_contract.json")
    summary = _read(parent / "comparison_summary.json")
    b0_original = _read(parent / "b0_256_original" / "completed.json")
    candidate = _read(parent / TRIAL / "completed.json")
    _check(names == contract["class_names"] == CLASSES, "클래스 순서가 다릅니다.")
    _check(config["settings"]["dataset_sha256"] == DATA_SHA256, "데이터 버전이 다릅니다.")
    _check(config["signature"] == candidate["signature"], "실험 설정이 다릅니다.")
    _check(summary["winner"] == "b1_384_original", "기존 성능 우선 후보가 다릅니다.")
    _check(summary["selected_model_sha256"] == HEAVY_SHA256, "기존 성능 우선 모델이 다릅니다.")
    _check(
        candidate["validation"]["macro_f1"] > b0_original["validation"]["macro_f1"],
        "B0 경량 후보의 Validation 선정 근거가 다릅니다.",
    )
    _check(
        candidate["id"] == TRIAL
        and candidate["selected_stage"] == "stage2"
        and candidate["selected_model"] == "stage2_best.keras"
        and candidate["spec"]
        == {
            "id": TRIAL,
            "backbone": "B0",
            "size": 256,
            "loss": "ce",
            "variant": "augmented",
            "class_count": 6,
        },
        "B0 후보 학습 조건이 다릅니다.",
    )
    relative = f"{candidate['attempt']}/{candidate['selected_model']}"
    model_path = parent / TRIAL / relative
    _check(model_path.is_file(), "B0 선정 모델 파일이 없습니다.")
    _check(
        _sha(model_path) == candidate["artifact_hashes"][relative] == MODEL_SHA256,
        "B0 선정 모델 해시가 다릅니다.",
    )
    _check(model_path.stat().st_size == candidate["model_bytes"], "B0 모델 크기가 다릅니다.")
    return {
        "model_path": model_path,
        "candidate": candidate,
        "contract": contract,
        "config": config,
        "summary": summary,
        "classes": names,
    }


def _verify_model(path: Path):
    import keras
    import numpy as np

    keras.backend.clear_session()
    model = keras.models.load_model(path, compile=False)
    _check(tuple(model.input_shape) == (None, 256, 256, 3), "B0 입력 크기가 다릅니다.")
    _check(tuple(model.output_shape) == (None, 6), "B0 클래스 수가 다릅니다.")
    _check(model.count_params() == 4057257, "B0 파라미터 수가 다릅니다.")
    backbones = [
        layer
        for layer in model.layers
        if isinstance(layer, keras.Model) and "efficientnetb0" in layer.name.lower()
    ]
    _check(len(backbones) == 1, "EfficientNet-B0 본체를 찾을 수 없습니다.")
    rescaling = [
        layer for layer in backbones[0].layers if isinstance(layer, keras.layers.Rescaling)
    ]
    _check(
        any(
            np.isclose(float(np.asarray(layer.scale).item()), 1 / 255)
            and np.allclose(layer.offset, 0)
            for layer in rescaling
        ),
        "B0 내부 픽셀 변환이 다릅니다.",
    )
    pixels = np.broadcast_to(
        np.array([0, 127.5, 255], dtype="float32")[:, None, None, None],
        (3, 256, 256, 3),
    ).copy()
    scores = np.asarray(model(pixels, training=False))
    _check(
        scores.shape == (3, 6)
        and np.isfinite(scores).all()
        and np.all((scores >= 0) & (scores <= 1))
        and np.allclose(scores.sum(axis=1), 1, atol=1e-5),
        "B0 예측 출력이 올바르지 않습니다.",
    )
    return model


def _extract_test(dataset_zip: Path, destination: Path, contract: dict):
    # The dataset builder archived its local `dataset/` directory as ZIP root.
    prefix = "dataset"
    expected = contract["original_counts"]["test"]
    entries = []
    with zipfile.ZipFile(dataset_zip) as archive:
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            _check(
                not path.is_absolute() and ".." not in path.parts and "\\" not in info.filename,
                "데이터 ZIP 경로가 안전하지 않습니다.",
            )
            if (
                not info.is_dir()
                and len(path.parts) == 5
                and path.parts[:3] == (prefix, "original", "test")
                and path.parts[3] in CLASSES
                and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            ):
                entries.append((info, path.parts[3], path))
        counts = {name: sum(label == name for _, label, _ in entries) for name in CLASSES}
        _check(counts == expected and sum(counts.values()) == 1305, "Test 분할 구성이 다릅니다.")
        for info, _label, path in entries:
            target = destination.joinpath(*path.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as sink:
                shutil.copyfileobj(source, sink)
    return destination / prefix


def _test_dataset(root: Path, batch_size: int):
    import numpy as np
    import tensorflow as tf

    paths = [
        path
        for name in CLASSES
        for path in sorted((root / "original" / "test" / name).iterdir())
        if path.is_file()
    ]
    labels = [CLASSES.index(path.parent.name) for path in paths]
    relative = [path.relative_to(root).as_posix() for path in paths]
    dataset = tf.data.Dataset.from_tensor_slices(
        ([str(path) for path in paths], np.asarray(labels, dtype=np.int32))
    )

    def decode(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape((None, None, 3))
        image = tf.image.resize(tf.cast(image, tf.float32), (256, 256))
        return image, tf.one_hot(label, 6)

    return dataset.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(
        tf.data.AUTOTUNE
    ), relative


def _evaluate_once(records: dict, dataset_zip: Path, output: Path, batch_size: int):
    from mediflow_datasets import common_engine as engine

    marker = output / "test_completed.json"
    if marker.exists():
        completed = _read(marker)
        _check(
            completed["model_sha256"] == MODEL_SHA256 and completed["data_sha256"] == DATA_SHA256,
            "저장된 Test 평가의 모델·데이터가 다릅니다.",
        )
        for filename, digest in completed["artifact_hashes"].items():
            _check(_sha(output / filename) == digest, f"기존 Test 결과가 변경됐습니다: {filename}")
        return completed["metrics"]
    _check(_sha(dataset_zip) == DATA_SHA256, "데이터 ZIP SHA-256이 다릅니다.")
    with tempfile.TemporaryDirectory(prefix="hair6_light_test_") as temporary:
        root = _extract_test(dataset_zip, Path(temporary), records["contract"])
        dataset, paths = _test_dataset(root, batch_size)
        model = _verify_model(records["model_path"])
        truth, probabilities = engine.predict_dataset(model, dataset)
        del model
        metrics = engine.classification_metrics(truth, probabilities, 6)
        metrics["selected_model"] = TRIAL
        engine.save_predictions(output / "test_predictions.csv", paths, truth, probabilities)
    _check(
        metrics["support"] == list(records["contract"]["original_counts"]["test"].values()),
        "평가 클래스별 장수가 다릅니다.",
    )
    _write(output / "test_metrics.json", metrics)
    completed = {
        "model_sha256": MODEL_SHA256,
        "data_sha256": DATA_SHA256,
        "metrics": metrics,
        "artifact_hashes": {
            name: _sha(output / name) for name in ("test_metrics.json", "test_predictions.csv")
        },
    }
    _write(marker, completed)
    return metrics


def package_light(parent_run_dir: str, project_root: str, batch_size: int = 32):
    """Read-only source check, one fixed Test evaluation, then separate package ZIP."""
    parent = Path(parent_run_dir)
    project = Path(project_root)
    records = verify_candidate(parent)
    dataset_zip = project / "datasets" / DATA_NAME
    _check(dataset_zip.is_file(), f"데이터 ZIP을 찾을 수 없습니다: {dataset_zip}")
    target_parent = project / "2_results" / "hair" / "selected_models"
    target_parent.mkdir(parents=True, exist_ok=True)
    output = target_parent / OUTPUT_NAME
    output.mkdir(exist_ok=True)
    archive = target_parent / f"{OUTPUT_NAME}.zip"
    if archive.exists():
        manifest = _read(output / "manifest.json")
        _check(
            all(_sha(output / name) == digest for name, digest in manifest.items()),
            "기존 경량 패키지 파일이 변경됐습니다.",
        )
        with zipfile.ZipFile(archive) as existing:
            _check(existing.testzip() is None, "기존 ZIP이 손상됐습니다.")
        return output, archive, _read(output / "test_metrics.json")
    metrics = _evaluate_once(records, dataset_zip, output, batch_size)
    sources = {
        "hair_model.keras": records["model_path"],
        "class_names.json": parent / "class_names.json",
        "dataset_contract.json": parent / "dataset_contract.json",
        "run_config.json": parent / "run_config.json",
        "comparison_summary.json": parent / "comparison_summary.json",
        "winner_completed.json": parent / TRIAL / "completed.json",
        "winner_validation_metrics.json": parent
        / TRIAL
        / records["candidate"]["attempt"]
        / "validation_metrics.json",
        "four_training_curves.png": parent / "four_training_curves.png",
        "validation_dashboard.png": parent / "validation_dashboard.png",
    }
    for name, source in sources.items():
        _check(source.is_file(), f"패키지 입력 파일이 없습니다: {source}")
        shutil.copyfile(source, output / name)
        _check(_sha(output / name) == _sha(source), f"복사 검증 실패: {name}")
    _write(
        output / "preprocessing.json",
        {
            "input_shape": [256, 256, 3],
            "color_order": "RGB",
            "input_dtype": "float32",
            "input_pixel_range": [0, 255],
            "external_normalization": False,
            "internal_rescaling": "1/255",
            "resize": "TensorFlow bilinear, antialias=False",
            "aspect_ratio": "direct resize, no crop/pad",
            "exif_transpose": False,
            "output": "six softmax scores in class_names.json order",
        },
    )
    _write(
        output / "model_card.json",
        {
            "version": "hair_6class_light_v1",
            "role": "lightweight_alternate_for_cost_comparison",
            "domain": "hair",
            "status": "public_data_candidate_not_device_validated",
            "backbone": "EfficientNet-B0",
            "input_size": [256, 256],
            "train_variant": "augmented",
            "loss": "categorical crossentropy",
            "stage1_epochs": 15,
            "stage2_epochs": 15,
            "classes": CLASSES,
            "model_sha256": MODEL_SHA256,
            "data_zip_sha256": DATA_SHA256,
            "validation": records["candidate"]["validation"],
            "test": metrics,
            "comparison_model_sha256": HEAVY_SHA256,
            "limitations": records["contract"]["limitations"]
            + [
                "Actual USB microscope patient images were not evaluated.",
                "Latency, energy and peak memory have not been measured on deployment hardware.",
                "Softmax scores are not calibrated correctness probabilities.",
            ],
        },
    )
    (output / "MODEL_INFO.md").write_text(
        "# Hair 6클래스 경량 비교 후보\n\n"
        "B0·256·원본+증강·CE. 두 B0 실험 중 Validation Macro F1이 더 높아 "
        "경량 후보로 고정했습니다. 성능 우선 B1·384·원본 모델은 별도로 보존합니다.\n\n"
        "입력: 두피 RGB 사진을 256×256으로 직접 조정한 float32 픽셀 0–255. "
        "외부 /255 정규화는 하지 않습니다.\n\n"
        f"클래스 출력 순서: {', '.join(CLASSES)}\n\n"
        f"Validation Accuracy {records['candidate']['validation']['accuracy']}; "
        f"Macro F1 {records['candidate']['validation']['macro_f1']}\n"
        f"Test Accuracy {metrics['accuracy']}; Macro F1 {metrics['macro_f1']} "
        f"(n={metrics['count']})\n\n"
        "경량 여부는 모델 파일 크기·입력 크기에 근거한 구분입니다. "
        "실제 장비의 지연 시간과 메모리는 아직 측정하지 않았습니다.\n",
        encoding="utf-8",
    )
    manifest = {path.name: _sha(path) for path in sorted(output.iterdir()) if path.is_file()}
    _write(output / "manifest.json", manifest)
    with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as target:
        for path in sorted(output.iterdir()):
            target.write(path, f"{OUTPUT_NAME}/{path.name}")
    with zipfile.ZipFile(archive) as target:
        _check(target.testzip() is None, "새 경량 ZIP이 손상됐습니다.")
    (target_parent / f"{OUTPUT_NAME}.zip.sha256").write_text(
        f"{_sha(archive)}  {archive.name}\n", encoding="utf-8"
    )
    return output, archive, metrics


In [ ]:
import sys, types
SOURCES = {
    'common_engine': (_EMBEDDED_SOURCE_DIR / 'common_engine.py').read_text(encoding='utf-8'),
    'hair_six_class_light': (_EMBEDDED_SOURCE_DIR / 'hair_six_class_light.py').read_text(encoding='utf-8'),
}
import hashlib as _source_hashlib
_EXPECTED_SOURCE_SHA256 = {'common_engine': '9bb8b0bbbc1e293d45792a8263903f322051aac1209c536c8dbc11a40c7b1038', 'hair_six_class_light': 'f9f82a89ee6db7a11be58df7ac1ddf011ae3a64dbd21012f6d0bdb68bc2e18c8'}
for _name, _code in SOURCES.items():
    if _source_hashlib.sha256(_code.encode('utf-8')).hexdigest() != _EXPECTED_SOURCE_SHA256[_name]:
        raise ValueError(f'내장 코드가 원본과 다릅니다: {_name}. 셀을 다시 실행하세요.')
package_module = types.ModuleType('mediflow_datasets')
package_module.__path__ = []
sys.modules['mediflow_datasets'] = package_module
for name, source in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets.hair_six_class_light import package_light, verify_candidate


## 4. 경량 후보 확인

모델 해시와 Validation 선정 근거를 먼저 확인합니다.


In [ ]:
from pathlib import Path
candidate = verify_candidate(Path(PARENT_RUN_DIR))
print('모델:', candidate['model_path'])
print('Validation Accuracy:', candidate['candidate']['validation']['accuracy'])
print('Validation Macro F1:', candidate['candidate']['validation']['macro_f1'])


## 5. Test 1회 평가와 패키징

Test 사진은 실행 중 임시 공간에만 풀고, 평가 뒤 제거합니다. 결과는 Drive의 `2_results/hair/selected_models`에 남습니다. 중단 후 다시 실행하면 완료된 평가의 해시를 확인하고 재사용합니다.


In [ ]:
output, model_zip, metrics = package_light(
    PARENT_RUN_DIR, PROJECT_ROOT, batch_size=BATCH_SIZE
)
print('경량 후보 ZIP:', model_zip)
print('ZIP SHA-256 파일:', model_zip.with_suffix('.zip.sha256'))
print('Test Accuracy:', metrics['accuracy'])
print('Test Macro F1:', metrics['macro_f1'])
print('모델 크기(MiB):', round((output / 'hair_model.keras').stat().st_size / 1024**2, 1))
